In [1]:
!pip install langchain langchain-google-genai sentence-transformers faiss-cpu


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: C:\Users\ASUS\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [2]:
import json
import os
import requests
from typing import List, Dict
from langchain_openai import ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
llm = ChatOpenAI(model="qwen/qwen-2.5-72b-instruct", temperature=0.1)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [14]:
class SimplePRLoader:
    
    def __init__(self, json_file_path: str):
        self.json_file_path = json_file_path
        self.pr_data = []
        
    def load_data(self, limit: int = None):
        print(f"📂 Loading PR data from {self.json_file_path}...")
        
        try:
            with open(self.json_file_path, 'r', encoding='utf-8') as f:
                all_data = json.load(f)
            
            # Apply limit only if specified
            if limit:
                self.pr_data = all_data[:limit]
                print(f"✅ Loaded {len(self.pr_data)} PRs (limited from {len(all_data)} total)")
            else:
                self.pr_data = all_data
                print(f"✅ Loaded ALL {len(self.pr_data)} PRs")
            
            return True
            
        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return False
    
    def get_pr_summary(self, pr: Dict) -> str:
        author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
        
        # Get repository and project info
        repo_name = pr.get('repo_name', 'Unknown Repository')
        
        # Get changed files info
        changed_files = pr.get('changed_files', [])
        changed_files_list = []
        if changed_files:
            for file_info in changed_files[:5]:  # Show first 5 files
                if isinstance(file_info, dict):
                    filename = file_info.get('filename', 'unknown')
                    status = file_info.get('status', 'modified')
                    changed_files_list.append(f"  - {filename} ({status})")
                else:
                    changed_files_list.append(f"  - {file_info}")
        
        # Get language statistics
        lang_stats = pr.get('language_stats', {})
        languages = ', '.join([f"{lang}: {lines}" for lang, lines in lang_stats.items()]) if lang_stats else 'Not specified'
        
        # Format created and updated dates
        created_at = pr.get('created_at', 'Unknown')
        updated_at = pr.get('updated_at', 'Unknown')
        
        summary = f"""
REPOSITORY: {repo_name}
PR #{pr.get('pr_number', 'N/A')}: {pr.get('title', 'No title')}
Author: {author}
State: {pr.get('state', 'unknown')}
Is Draft: {pr.get('is_draft', False)}
Created: {created_at}
Updated: {updated_at}

DESCRIPTION:
{pr.get('description', 'No description')}

CHANGE STATISTICS:
- Files Changed: {pr.get('changed_files_count', 0)}
- Additions: +{pr.get('additions', 0)} lines
- Deletions: -{pr.get('deletions', 0)} lines
- Commits: {pr.get('commits_count', 0)}
- Comments: {pr.get('comments_count', 0)}
- Review Comments: {pr.get('review_comments_count', 0)}

LANGUAGES INVOLVED:
{languages}

LABELS: {', '.join(pr.get('labels', [])) if pr.get('labels') else 'None'}

CHANGED FILES:
{chr(10).join(changed_files_list) if changed_files_list else 'No file details available'}

        DIFF URL: {pr.get('diff_url', 'Not available')}
PATCH URL: {pr.get('patch_url', 'Not available')}
"""
        return summary.strip()
    
    def get_code_changes(self, pr: Dict, max_size: int = 10000) -> str:
        """Fetch actual code changes from diff URL"""
        diff_url = pr.get('diff_url')
        if not diff_url:
            return "No diff URL available"
        
        try:
            print(f"📥 Fetching code changes from {diff_url}...")
            response = requests.get(diff_url, timeout=10)
            
            if response.status_code == 200:
                diff_content = response.text
                
                # Limit size to avoid token limits
                if len(diff_content) > max_size:
                    diff_content = diff_content[:max_size] + "\n\n... [TRUNCATED - Diff too large] ..."
                
                return diff_content
            else:
                return f"Failed to fetch diff (HTTP {response.status_code})"
                
        except requests.RequestException as e:
            return f"Error fetching diff: {str(e)}"
        except Exception as e:
            return f"Unexpected error: {str(e)}"

# Load ALL PR data from multiple repositories
pr_data_files = [
     "PR data for Developers/express_test_data.json",
]

print("🚀 Loading ALL PR data from multiple repositories...")
all_pr_data = []

for file_path in pr_data_files:
    try:
        print(f"📂 Loading from {file_path}...")
        with open(file_path, 'r', encoding='utf-8') as f:
            repo_data = json.load(f)
        
        # Add repository identifier to each PR
        repo_name = file_path.split('/')[0].replace(' PR data', '')
        for pr in repo_data:
            pr['source_repo'] = repo_name
        
        all_pr_data.extend(repo_data)
        print(f"✅ Loaded {len(repo_data)} PRs from {repo_name}")
        
    except FileNotFoundError:
        print(f"⚠️ File not found: {file_path} - Skipping...")
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")

print(f"\n🎉 TOTAL: Loaded {len(all_pr_data)} PRs from all repositories!")

# Create PR loader with all data
pr_loader = SimplePRLoader("PR data for Developers/express_test_data.json")
pr_loader.pr_data = all_pr_data  # Directly assign all loaded data
success = True

🚀 Loading ALL PR data from multiple repositories...
📂 Loading from PR data for Developers/express_test_data.json...
✅ Loaded 627 PRs from PR data for Developers

🎉 TOTAL: Loaded 627 PRs from all repositories!


In [15]:
class SimpleVectorStore:
    
    def __init__(self, embeddings_model):
        self.embeddings = embeddings_model
        self.vector_store = None
        self.documents = []
    
    def create_embeddings(self, pr_data: List[Dict]):
        """Convert PR data to vector embeddings"""
        print("🔄 Creating vector embeddings...")
        
        documents = []
        
        # Convert each PR to a document
        for pr in pr_data:
            # Create text content with repository info
            author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
            repo_name = pr.get('repo_name', 'Unknown Repository')
            
            # Get file information
            changed_files = pr.get('changed_files', [])
            file_names = []
            if changed_files:
                for file_info in changed_files[:10]:  # First 10 files
                    if isinstance(file_info, dict):
                        file_names.append(file_info.get('filename', 'unknown'))
                    else:
                        file_names.append(str(file_info))
            
            # Get language statistics
            lang_stats = pr.get('language_stats', {})
            languages = ', '.join(lang_stats.keys()) if lang_stats else ''
            
            content = f"""
Repository: {repo_name}
PR #{pr.get('pr_number')}: {pr.get('title', '')}
Author: {author}
Description: {pr.get('description', '')}
Labels: {', '.join(pr.get('labels', []))}
State: {pr.get('state', 'unknown')}
Is Draft: {pr.get('is_draft', False)}
Files: {pr.get('changed_files_count', 0)} changed
Changes: +{pr.get('additions', 0)} -{pr.get('deletions', 0)}
Languages: {languages}
Changed Files: {', '.join(file_names[:5])}
"""
            
            # Create metadata
            metadata = {
                'pr_number': pr.get('pr_number'),
                'author': author,
                'state': pr.get('state'),
                'title': pr.get('title', ''),
                'repo_name': repo_name,
                'additions': pr.get('additions', 0),
                'deletions': pr.get('deletions', 0),
                'changed_files_count': pr.get('changed_files_count', 0)
            }
            
            # Create document
            doc = Document(page_content=content.strip(), metadata=metadata)
            documents.append(doc)
        
        # Create vector store
        self.vector_store = FAISS.from_documents(documents, self.embeddings)
        self.documents = documents
        
        print(f"✅ Created {len(documents)} vector embeddings")
        return self.vector_store
    
    def find_similar_prs(self, query: str, k: int = 5) -> List[Document]:
        """Find similar PRs using vector search"""
        if not self.vector_store:
            print("❌ Vector store not created yet!")
            return []
        
        # Perform similarity search
        similar_docs = self.vector_store.similarity_search(query, k=k)
        return similar_docs

# Create vector store
vector_store = SimpleVectorStore(embeddings)
if pr_loader.pr_data:
    vector_store.create_embeddings(pr_loader.pr_data)
    print("🎯 Vector store ready for similarity search!")
else:
    print("❌ No PR data available for embedding")

🔄 Creating vector embeddings...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ Created 627 vector embeddings
🎯 Vector store ready for similarity search!


In [25]:
import glob
from typing import List, Dict, Tuple

class SmartPRAssigner:
    """Enhanced PR reviewer that can assign PRs to the most suitable developers"""
    
    def __init__(self, llm_model, vector_store, profiles_directory: str = "generated_profiles"):
        self.llm = llm_model
        self.vector_store = vector_store
        self.profiles_directory = profiles_directory
        self.developer_profiles = {}
        self.load_developer_profiles()
        
        # Prompt for analyzing PR requirements
        self.analysis_prompt = PromptTemplate(
            input_variables=["pr_content", "code_changes"],
            template="""You are an expert technical analyst. Analyze this Pull Request and identify the key technical skills and expertise areas needed to review it effectively.


🎯 CRITICAL: Pay special attention to FREQUENCY DATA in developer profiles. 
Higher frequency numbers (e.g., "Express.js (10x)") indicate MORE REAL EXPERIENCE with that technology.
Prioritize developers with HIGH FREQUENCY in the required skills over those with low/no frequency data.

PULL REQUEST TO ANALYZE:
{pr_content}

CODE CHANGES:
{code_changes}

Based on this PR, identify:
1. **Primary Technical Skills Needed**: What specific technologies, frameworks, or languages are involved?
2. **Expertise Areas Required**: What domain knowledge is needed (e.g., frontend, backend, testing, security, performance)?
3. **Complexity Level**: How complex is this change? (Low/Medium/High)
4. **Review Focus Areas**: What should the reviewer pay special attention to?

🎯 ASSIGNMENT STRATEGY - Use frequency data to make better decisions:
1. **Frequency Matching**: Prioritize developers with HIGH frequency numbers in required skills
   - "Express.js (10x)" is better than "Express.js (2x)" 
   - "Node.js (15x)" indicates extensive real-world experience
2. **Experience Level**: Match developer experience to PR complexity
3. **Skill Relevance**: Look for direct matches in required technologies

Provide your analysis in this exact JSON format:
{{
    "technical_skills_needed": ["skill1", "skill2", "skill3"],
    "expertise_areas_needed": ["area1", "area2"],
    "complexity_level": "Low/Medium/High",
    "review_focus_areas": ["focus1", "focus2", "focus3"],
    "primary_language": "JavaScript/Python/etc",
    "frameworks_involved": ["framework1", "framework2"]
}}

ANALYSIS:"""
        )
        
        # Prompt for matching developers
        self.matching_prompt = PromptTemplate(
            input_variables=["pr_requirements", "developer_profiles"],
            template="""You are an expert at matching technical requirements with developer expertise. 

PR REQUIREMENTS:
{pr_requirements}

AVAILABLE DEVELOPERS:
{developer_profiles}

Your task is to rank the top 3 developers who would be best suited to review this PR, considering:
1. Technical skill alignment with PR requirements
2. Relevant expertise areas
3. Experience level appropriateness for the complexity
4. Past contribution patterns

Provide your recommendation in this exact JSON format:
{{
    "recommended_developers": [
        {{
            "developer_name": "developer1",
            "match_score": 0.95,
            "reasoning": "Why this developer is the best match",
            "strengths_alignment": ["strength1", "strength2"],
            "potential_concerns": "Any concerns or gaps"
        }}
    ],
    "assignment_confidence": "High/Medium/Low",
    "assignment_reasoning": "Overall reasoning for the recommendations"
}}

RECOMMENDATION:"""
        )
        
        self.analysis_chain = LLMChain(llm=self.llm, prompt=self.analysis_prompt)
        self.matching_chain = LLMChain(llm=self.llm, prompt=self.matching_prompt)
    
    def load_developer_profiles(self):
        """Load all developer profiles from JSON files"""
        profile_files = glob.glob(f"{self.profiles_directory}/*.json")
        
        print(f"🔍 Loading developer profiles from {self.profiles_directory}/...")
        
        for file_path in profile_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    profile_data = json.load(f)
                
                dev_name = profile_data['metadata']['developer_name']
                self.developer_profiles[dev_name] = profile_data['profile']
                
            except Exception as e:
                print(f"❌ Error loading profile {file_path}: {e}")
        
        print(f"✅ Loaded {len(self.developer_profiles)} developer profiles")
        print(f"👥 Available developers: {', '.join(self.developer_profiles.keys())}")
    
    def display_developer_details(self, developer_name: str):
        """Display detailed information about a specific developer"""
        if developer_name not in self.developer_profiles:
            print(f"❌ Developer '{developer_name}' not found")
            return
        
        profile = self.developer_profiles[developer_name]
        print(f"\n👨‍💻 DEVELOPER PROFILE: {developer_name}")
        print("=" * 50)
        print(f"Experience Level: {profile.get('experience_level', 'Unknown')}")
        print(f"Languages: {', '.join(profile.get('programming_languages', []))}")
        print(f"Primary Skills: {', '.join(profile.get('primary_skills', []))}")
        
        # Show top skills by frequency - extract from JavaScript skill matrix
        if 'javascript_skill_matrix' in profile:
            print("\nTop Skills by Experience:")
            js_matrix = profile['javascript_skill_matrix']
            skill_freq_pairs = []
            
            # Extract skills and frequencies from the matrix
            for category, skills in js_matrix.items():
                if skills:
                    for skill in skills:
                        # Extract frequency from skill string (e.g., "Express.js, frequency: 10")
                        if 'frequency:' in skill:
                            parts = skill.split(', frequency:')
                            if len(parts) == 2:
                                skill_name = parts[0].strip()
                                freq = int(parts[1].strip())
                                skill_freq_pairs.append((skill_name, freq))
            
            # Sort by frequency and display top 5
            sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
            for skill, freq in sorted_skills[:5]:
                print(f"  • {skill}: {freq} PRs")
        
        # Show JavaScript expertise matrix
        if 'javascript_skill_matrix' in profile:
            print("\nJavaScript Expertise Areas:")
            js_matrix = profile['javascript_skill_matrix']
            for category, skills in js_matrix.items():
                if skills:
                    # Clean category name (remove frequency info)
                    clean_category = category.split(', frequency:')[0]
                    print(f"  {clean_category}:")
                    for skill in skills[:3]:  # Top 3 in each category
                        # Clean skill name (remove frequency info)
                        clean_skill = skill.split(', frequency:')[0]
                        print(f"    - {clean_skill}")
        
        print(f"\nSummary: {profile.get('summary', 'No summary available')}")
        print("=" * 50)
    
    def analyze_pr_requirements(self, pr_data: Dict) -> Dict:
        """Analyze what technical skills and expertise this PR requires"""
        
        # Get PR summary and code changes
        pr_content = pr_loader.get_pr_summary(pr_data)
        code_changes = pr_loader.get_code_changes(pr_data, max_size=5000)  # Smaller for analysis
        
        try:
            # Analyze PR requirements
            analysis_result = self.analysis_chain.run(
                pr_content=pr_content,
                code_changes=code_changes
            )
            
            # Try to parse JSON response
            import re
            json_match = re.search(r'\{.*\}', analysis_result, re.DOTALL)
            if json_match:
                requirements = json.loads(json_match.group())
                return requirements
            else:
                print("⚠️ Could not parse PR analysis JSON, using fallback")
                return self._fallback_analysis(pr_data)
                
        except Exception as e:
            print(f"❌ Error analyzing PR requirements: {e}")
            return self._fallback_analysis(pr_data)
    
    def _fallback_analysis(self, pr_data: Dict) -> Dict:
        """Fallback analysis based on PR metadata"""
        changed_files = pr_data.get('changed_files', [])
        language_stats = pr_data.get('language_stats', {})
        
        # Determine primary language
        primary_lang = "JavaScript"  # Default for this dataset
        if language_stats:
            primary_lang = max(language_stats.keys(), key=lambda k: language_stats[k])
        
        # Determine expertise areas based on file patterns
        expertise_areas = ["General Development"]
        if any('test' in str(f).lower() for f in changed_files):
            expertise_areas.append("Testing")
        if any('doc' in str(f).lower() for f in changed_files):
            expertise_areas.append("Documentation")
        
        return {
            "technical_skills_needed": [primary_lang, "Git"],
            "expertise_areas_needed": expertise_areas,
            "complexity_level": "Medium",
            "review_focus_areas": ["Code Quality", "Functionality"],
            "primary_language": primary_lang,
            "frameworks_involved": ["Unknown"]
        }
    
    def find_best_reviewers(self, pr_requirements: Dict, top_k: int = 3) -> Dict:
        """Find the best developers to review this PR"""
        
        # Format developer profiles for the LLM
        profiles_summary = ""
        for dev_name, profile in self.developer_profiles.items():
            # Extract top skills from JavaScript skill matrix if available
            top_skills = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                skill_freq_pairs = []
                
                # Extract skills and frequencies from the matrix
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_freq_pairs.append((skill_name, freq))
                
                # Sort by frequency and get top 5
                sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
                top_skills = [skill for skill, freq in sorted_skills[:5]]
            
            # Fallback to primary skills if no frequency data
            if not top_skills:
                top_skills = profile.get('primary_skills', [])
            
            # Get JavaScript-specific expertise if available
            js_expertise = ""
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                # Look for framework expertise (clean the category names)
                framework_expertise = []
                backend_skills = []
                
                for category, skills in js_matrix.items():
                    clean_category = category.split(', frequency:')[0].lower()
                    if 'framework' in clean_category or 'library' in clean_category:
                        framework_expertise = [skill.split(', frequency:')[0] for skill in skills]
                    elif 'backend' in clean_category:
                        backend_skills = [skill.split(', frequency:')[0] for skill in skills]
                
                if framework_expertise or backend_skills:
                    js_expertise = f"\n- JavaScript Expertise: {', '.join(framework_expertise[:3] + backend_skills[:3])}"
            
            profiles_summary += f"""
DEVELOPER: {dev_name}
- Experience Level: {profile.get('experience_level', 'Unknown')}
- Top Skills (by frequency): {', '.join(top_skills)}
- Programming Languages: {', '.join(profile.get('programming_languages', []))}
- Primary Skills: {', '.join(profile.get('primary_skills', []))}{js_expertise}
- Summary: {profile.get('summary', 'No summary available')[:200]}...

"""
        
        try:
            # Get matching recommendations
            matching_result = self.matching_chain.run(
                pr_requirements=json.dumps(pr_requirements, indent=2),
                developer_profiles=profiles_summary
            )
            
            # Try to parse JSON response
            import re
            json_match = re.search(r'\{.*\}', matching_result, re.DOTALL)
            if json_match:
                recommendations = json.loads(json_match.group())
                return recommendations
            else:
                print("⚠️ Could not parse matching JSON, using fallback")
                return self._fallback_matching(pr_requirements)
                
        except Exception as e:
            print(f"❌ Error finding best reviewers: {e}")
            return self._fallback_matching(pr_requirements)
    
    def _fallback_matching(self, pr_requirements: Dict) -> Dict:
        """Enhanced fallback matching using skill frequency data from JavaScript matrix"""
        primary_lang = pr_requirements.get('primary_language', 'JavaScript')
        needed_skills = pr_requirements.get('technical_skills_needed', [])
        needed_frameworks = pr_requirements.get('frameworks_involved', [])
        
        # Enhanced scoring based on skill frequency and expertise
        developer_scores = []
        for dev_name, profile in self.developer_profiles.items():
            score = 0.0
            
            # Get all available skills
            dev_skills = profile.get('primary_skills', []) + profile.get('programming_languages', [])
            
            # Extract skill frequencies from JavaScript skill matrix
            skill_frequencies = {}
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_frequencies[skill_name] = freq
            
            # Language match with frequency bonus
            if primary_lang in profile.get('programming_languages', []):
                base_score = 0.4
                # Bonus for high frequency of that language
                if primary_lang in skill_frequencies:
                    frequency_bonus = min(skill_frequencies[primary_lang] / 20, 0.2)  # Max 0.2 bonus
                    score += base_score + frequency_bonus
                else:
                    score += base_score
            
            # Skill overlap with frequency weighting
            for skill in needed_skills:
                if skill in skill_frequencies:
                    # Weight by skill frequency (more experience = higher score)
                    frequency_weight = min(skill_frequencies[skill] / 10, 0.3)  # Max 0.3 per skill
                    score += frequency_weight
                elif skill in dev_skills:
                    score += 0.1  # Basic skill match
            
            # Framework expertise bonus
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                # Look for framework expertise in any category containing "framework" or "library"
                for category, skills in js_matrix.items():
                    clean_category = category.split(', frequency:')[0].lower()
                    if 'framework' in clean_category or 'library' in clean_category:
                        framework_expertise = [skill.split(', frequency:')[0] for skill in skills]
                        for framework in needed_frameworks:
                            if any(framework.lower() in expertise.lower() for expertise in framework_expertise):
                                score += 0.2
            
            # Experience level bonus
            experience_level = profile.get('experience_level', 'Junior')
            complexity = pr_requirements.get('complexity_level', 'Medium')
            if (complexity == 'High' and experience_level in ['Senior', 'Expert']) or \
               (complexity == 'Medium' and experience_level in ['Mid', 'Senior', 'Expert']) or \
               (complexity == 'Low'):
                score += 0.1
            
            developer_scores.append((dev_name, score))
        
        # Sort by score and take top 3
        developer_scores.sort(key=lambda x: x[1], reverse=True)
        top_3 = developer_scores[:3]
        
        recommendations = []
        for i, (dev_name, score) in enumerate(top_3):
            profile = self.developer_profiles[dev_name]
            
            # Get top skills for this developer
            top_skills = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                skill_freq_pairs = []
                
                # Extract skills and frequencies from the matrix
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_freq_pairs.append((skill_name, freq))
                
                # Sort by frequency and get top 3
                sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
                top_skills = [skill for skill, freq in sorted_skills[:3]]
            
            # Fallback to primary skills
            if not top_skills:
                top_skills = profile.get('primary_skills', [])[:3]
            
            # Generate more detailed reasoning
            reasoning_parts = []
            if primary_lang in profile.get('programming_languages', []):
                reasoning_parts.append(f"Strong {primary_lang} experience")
            
            # Check for skill matches in the extracted frequencies
            skill_matches = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                for category, skills in js_matrix.items():
                    for skill in skills:
                        skill_name = skill.split(', frequency:')[0].strip()
                        if skill_name in needed_skills:
                            skill_matches.append(skill_name)
            
            if skill_matches:
                reasoning_parts.append(f"Experienced in {', '.join(skill_matches[:2])}")
            
            if not reasoning_parts:
                reasoning_parts.append("General development skills match")
            
            recommendations.append({
                "developer_name": dev_name,
                "match_score": min(score, 1.0),
                "reasoning": "; ".join(reasoning_parts),
                "strengths_alignment": top_skills,
                "potential_concerns": "Limited specific expertise" if score < 0.5 else "None identified"
            })
        
        confidence = "High" if top_3[0][1] > 0.7 else ("Medium" if top_3[0][1] > 0.4 else "Low")
        
        return {
            "recommended_developers": recommendations,
            "assignment_confidence": confidence,
            "assignment_reasoning": "Enhanced matching using skill frequency and expertise data"
        }
    
    def smart_review_and_assign(self, pr_data: Dict) -> Tuple[str, Dict]:
        """Complete workflow: analyze PR, find reviewers, and provide review"""
        
        print(f"🔍 Analyzing PR #{pr_data.get('pr_number')}: {pr_data.get('title')}")
        print("=" * 80)
        
        # Step 1: Analyze PR requirements
        print("📊 Step 1: Analyzing PR technical requirements...")
        pr_requirements = self.analyze_pr_requirements(pr_data)
        
        print(f"✅ Required Skills: {', '.join(pr_requirements.get('technical_skills_needed', []))}")
        print(f"✅ Expertise Areas: {', '.join(pr_requirements.get('expertise_areas_needed', []))}")
        print(f"✅ Complexity: {pr_requirements.get('complexity_level', 'Unknown')}")
        
        # Step 2: Find best reviewers
        print("\n👥 Step 2: Finding best-matched reviewers...")
        reviewer_recommendations = self.find_best_reviewers(pr_requirements)
        
        print("🎯 Recommended Reviewers:")
        for i, rec in enumerate(reviewer_recommendations.get('recommended_developers', []), 1):
            print(f"  {i}. {rec['developer_name']} (Score: {rec['match_score']:.2f})")
            print(f"     Reasoning: {rec['reasoning']}")
            print(f"     Strengths: {', '.join(rec.get('strengths_alignment', []))}")
            if rec.get('potential_concerns'):
                print(f"     Concerns: {rec['potential_concerns']}")
            print()
        
        # Step 3: Generate detailed review (using existing reviewer)
        print("📝 Step 3: Generating detailed PR review...")
        detailed_review = pr_reviewer.review_pr(pr_data, "comprehensive technical review")
        
        # Combine results
        assignment_summary = f"""
🎯 SMART PR ASSIGNMENT SUMMARY
{'=' * 50}

📋 PR ANALYSIS:
- Technical Skills Needed: {', '.join(pr_requirements.get('technical_skills_needed', []))}
- Expertise Areas: {', '.join(pr_requirements.get('expertise_areas_needed', []))}
- Complexity Level: {pr_requirements.get('complexity_level', 'Unknown')}
- Primary Language: {pr_requirements.get('primary_language', 'Unknown')}

👥 RECOMMENDED REVIEWERS:
"""
        
        for i, rec in enumerate(reviewer_recommendations.get('recommended_developers', []), 1):
            assignment_summary += f"""
{i}. **{rec['developer_name']}** (Match Score: {rec['match_score']:.2f})
   - Reasoning: {rec['reasoning']}
   - Key Strengths: {', '.join(rec.get('strengths_alignment', [])[:3])}
   - Concerns: {rec.get('potential_concerns', 'None identified')}
"""
        
        assignment_summary += f"""
🔍 Assignment Confidence: {reviewer_recommendations.get('assignment_confidence', 'Unknown')}
📝 Assignment Reasoning: {reviewer_recommendations.get('assignment_reasoning', 'Not provided')}

{'=' * 50}
"""
        
        return assignment_summary, {
            'pr_requirements': pr_requirements,
            'reviewer_recommendations': reviewer_recommendations,
            'detailed_review': detailed_review
        }

# Create the smart PR assigner
print("🚀 Creating Smart PR Assigner...")
smart_assigner = SmartPRAssigner(llm, vector_store, "Developer's Profiles")
print("✅ Smart PR Assigner ready!")

🚀 Creating Smart PR Assigner...
🔍 Loading developer profiles from Developer's Profiles/...
✅ Loaded 51 developer profiles
👥 Available developers: Abdel-Monaam-Aouini, agungjati, aheckmann, Ayoub-Mabrouk, bjohansebas, blakeembrey, buschtoens, carpasse, caub, ChALkeR, chenhaihong, crandmck, ctcpip, curious-attempt-bunny, czaarek99, davglass, defunctzombie, dougwilson, estrada9166, EvanHahn, frootloops, getspooky, ghost, gireeshpunathil, gmethvin, hacksparrow, Hashen110, hiroppy, howtoclient, IamLizu, jonathanong, jonchurch, jonjenkins, KoyamaSohei, madarche, mscdex, notrab, Phillip9587, raksbisht, rgrove, riadhchtara, rritik772, sakateka, sheplu, shivarm, tunniclm, Turbo87, UlisesGascon, vsopvsop, wesleytodd, ykumar6
✅ Smart PR Assigner ready!


In [26]:
# Quick Assignment Function - for easy testing with different PRs
def quick_assign_pr(pr_number: int):
    """Quick function to assign a PR by number"""
    # Find PR by number
    target_pr = None
    for pr in pr_loader.pr_data:
        if pr.get('pr_number') == pr_number:
            target_pr = pr
            break
    
    if not target_pr:
        print(f"❌ PR #{pr_number} not found in loaded data")
        return
    
    print(f"🎯 SMART ASSIGNMENT FOR PR #{pr_number}")
    print(f"📝 Title: {target_pr.get('title')}")
    print(f"👤 Author: {target_pr.get('author', {}).get('username', 'Unknown')}")
    print("=" * 60)
    
    # Get assignment recommendations
    pr_requirements = smart_assigner.analyze_pr_requirements(target_pr)
    reviewer_recommendations = smart_assigner.find_best_reviewers(pr_requirements)
    
    # Get the best developer (first in the list)
    recommended_devs = reviewer_recommendations.get('recommended_developers', [])
    if not recommended_devs:
        print("❌ No suitable developer found for this PR")
        return
    
    best_dev = recommended_devs[0]  # Get only the top recommendation
    
    print("🎯 ASSIGNED DEVELOPER:")
    print(f"\n👨‍💻 **{best_dev['developer_name']}** (Match: {best_dev['match_score']:.0%})")
    print(f"💡 Why: {best_dev['reasoning']}")
    print(f"⭐ Strengths: {', '.join(best_dev.get('strengths_alignment', [])[:3])}")
    if best_dev.get('potential_concerns'):
        print(f"⚠️ Concerns: {best_dev['potential_concerns']}")
    
    print(f"\n🔍 Assignment Confidence: {reviewer_recommendations.get('assignment_confidence', 'Unknown')}")
    print("=" * 60)

# Example usage - Test with a sample PR
print("🚀 EXAMPLE: Smart PR Assignment") 
quick_assign_pr(4262)

🚀 EXAMPLE: Smart PR Assignment
🎯 SMART ASSIGNMENT FOR PR #4262
📝 Title: Update response.js comment
👤 Author: kurtgalvin
📥 Fetching code changes from https://github.com/expressjs/express/pull/4262.diff...
🎯 ASSIGNED DEVELOPER:

👨‍💻 **Abdel-Monaam-Aouini** (Match: 95%)
💡 Why: Abdel-Monaam-Aouini has strong expertise in Node.js, Express.js, and backend development, which aligns perfectly with the PR requirements. They also have experience in code refactoring and security best practices, which are crucial for code review.
⭐ Strengths: Node.js, Express.js, Backend Development
⚠️ Concerns: None significant for this PR complexity level.

🔍 Assignment Confidence: High
